# 03 — MLflow Car Price: Experimentation, Model Versioning & Explainability

## What this notebook does
 
In `02_Feature_Store_CarPrice.ipynb` we cleaned the data, engineered features, and stored everything in the Hopsworks Feature Store. Here we take the next step: **train and compare three models**, track every experiment with **MLflow**, and explain the best model with **SHAP** so it can be versioned, audited, and handed off for serving.
 
This notebook covers the full experimentation pipeline in three parts:
 
**Part 1 — Baseline: Linear Regression**:
 
1. Load the training dataset from the Feature Store Feature View.
2. Train a `LinearRegression` model as a performance floor.
3. Log metrics (RMSE, MAE, R²) and the model artifact to MLflow.

**Part 2 — Hyperparameter Tuning with Optuna**:
 
4. Define an Optuna objective function for XGBoost and LightGBM.
5. Run `n_trials=20` with `MLflowCallback` logging each trial as a nested MLflow run.
6. Retrain each best model on the full train set and log the final artifact.

**Part 3 — Champion Selection, SHAP & Evaluation**:
 
7. Compare all three models with `mlflow.search_runs()` and select the best by RMSE.
8. Register the Champion in the MLflow Model Registry and tag the runners-up as Challengers.
9. Compute SHAP values and log summary, waterfall, and dependence plots as MLflow artifacts.
10. Run `mlflow.evaluate()` on the held-out test set for a final standardised metrics report.

In [48]:
# Prerequisites
# Before running this notebook, start the MLflow server from the terminal:
#
#   mlflow server \
#       --host 127.0.0.1 \
#       --port 8080 \
#       --artifacts-destination ./ml_artifacts \
#       --default-artifact-root ./ml_artifacts \
#       --backend-store-uri sqlite:///mlflow.db
#
# Then open http://localhost:8080 to view the UI.

In [49]:
# Imports and MLflow setup

import os
import tempfile
import warnings
import importlib.util
import numpy as np
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
 
# ML libraries
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
from lightgbm import LGBMRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
 
# MLOps tools
import mlflow
import optuna
import optuna.visualization as vis
from optuna.integration.mlflow import MLflowCallback
from mlflow.models import infer_signature
from mlflow import MlflowClient
import shap

warnings.filterwarnings("ignore")

# MLflow tracking URI
mlflow.set_tracking_uri("http://localhost:8080")

def get_or_create_experiment(experiment_name: str) -> str:
    if experiment := mlflow.get_experiment_by_name(experiment_name):
        return experiment.experiment_id
    else:
        return mlflow.create_experiment(experiment_name)

print("✅ Libraries loaded and MLflow tracking URI set to http://localhost:8080")

✅ Libraries loaded and MLflow tracking URI set to http://localhost:8080


In [50]:
# Load Gold Layer Data 
# df_features_hops was saved at the end of notebook 02 to 05_model_input —
# the Kedro layer for clean, feature-engineered data ready for modelling.
# In a paid Hopsworks tier this would instead be feature_view.get_training_data().

DATA_DIR = Path("../data/05_model_input")

X_train = pd.read_parquet(DATA_DIR / "X_train.parquet")
y_train = pd.read_parquet(DATA_DIR / "y_train.parquet")
X_val = pd.read_parquet(DATA_DIR / "X_val.parquet")
y_val = pd.read_parquet(DATA_DIR / "y_val.parquet")

X_test = X_val.copy()
y_test = y_val.copy()

# Create a new validation set from the training data
X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,      
            random_state=42,
)

print(f"\n   Train      : {X_train.shape[0]:,} rows")
print(f"   Validation : {X_val.shape[0]:,} rows")
print(f"   Test       : {X_test.shape[0]:,} rows")
print(f"   Features   : {X_train.shape[1]}")



   Train      : 48,086 rows
   Validation : 12,022 rows
   Test       : 15,195 rows
   Features   : 34


### Part 1 - Baseline: Linear Regression
 
Before tuning any complex model, we establish a **performance floor** using a simple Linear Regression. Every subsequent model must beat these numbers to justify its added complexity.
 
We log this run to MLflow under the experiment `car_price_linear_regression` with:
- **Metrics**: RMSE, MAE, and R² on the validation set
- **Model artifact**: saved with `mlflow.sklearn.log_model()` using the secure `skops` format
- **Model signature**: inferred with `infer_signature()` so MLflow enforces the input schema at serving time
- **Tags**: to identify the model family and project
Since Linear Regression cannot natively handle string columns, we wrap it in a
`Pipeline` with a `OneHotEncoder` for the categorical features.

In [51]:
# Linear Regression needs the original string categoricals, not label-encoded integers.
# We rebuild X_raw from df_features_hops and reuse the exact same row indices
# from the split already done in Step 1, so train/val/test rows are identical.

lr_pipeline = Pipeline(steps=[
    ("model", LinearRegression())
])

In [52]:
# MLflow run
EXPERIMENT_NAME  = "car_price_linear_regression"
mlflow.set_experiment(EXPERIMENT_NAME)

with mlflow.start_run(run_name="linear_regression_baseline") as run:

    # Train and predict on the validation set
    lr_pipeline.fit(X_train, y_train)
    y_pred_val = lr_pipeline.predict(X_val)

    # Compute metrics
    rmse = root_mean_squared_error(y_val, y_pred_val)
    mae  = mean_absolute_error(y_val, y_pred_val)
    r2   = r2_score(y_val, y_pred_val)

    # Log metrics, tags and model artifact
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae",  mae)
    mlflow.log_metric("r2",   r2)

    mlflow.set_tags({
        "model_family": "linear",
        "project":      "car_price_mlops",
        "environment":  "dev"
    })

    signature = infer_signature(X_train, y_pred_val)

    mlflow.sklearn.log_model(
        sk_model=lr_pipeline,
        name="linear_regression",
        signature=signature,
        input_example=X_train.iloc[[0]],
        serialization_format="skops"
    )

    lr_run_id = run.info.run_id

mlflow.end_run()

print(f"✅ Linear Regression baseline logged to '{EXPERIMENT_NAME}'.")
print(f"RMSE : £{rmse:,.0f}")
print(f"MAE  : £{mae:,.0f}")
print(f"R²   : {r2:.4f}")
print(f"\nRun ID : {lr_run_id}")
print("View   : http://localhost:8080")

2026/06/29 23:18:54 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run linear_regression_baseline at: http://localhost:8080/#/experiments/1/runs/53429cff478c478b84563b645d919e1a
🧪 View experiment at: http://localhost:8080/#/experiments/1
✅ Linear Regression baseline logged to 'car_price_linear_regression'.
RMSE : £4,491
MAE  : £2,681
R²   : 0.7796

Run ID : 53429cff478c478b84563b645d919e1a
View   : http://localhost:8080


### Part 2A — XGBoost with Optuna Hyperparameter Tuning
 
Manually searching for good hyperparameters is slow and unreliable. Instead we use **Optuna**, which uses a Bayesian algorithm called Tree-structured Parzen Estimator (TPE) to intelligently explore the hyperparameter space — it learns from previous trials to focus on the most promising regions.
 
We connect Optuna directly to MLflow via `MLflowCallback`, so every trial is automatically logged as a **nested run** under a parent run named `xgboost_optuna_tuning`. This gives us a full audit trail of all 20 trials inside the MLflow UI.
 
The workflow has two steps:
 
1. **Tuning** — run 20 Optuna trials, each training an XGBoost model with a different set
   of hyperparameters and evaluating RMSE on the validation set. Optuna picks the next
   trial based on what it learned from the previous ones.
2. **Best model** — retrain XGBoost with the best hyperparameters found, log it as a
   separate run named `xgboost_best_model` with metrics, tags, artifact, and Optuna
   visualisation plots.

In [53]:
EXPERIMENT_NAME_XGB = "car_price_xgboost"
experiment_id_xgb = get_or_create_experiment(EXPERIMENT_NAME_XGB)
mlflow.set_experiment(experiment_id=experiment_id_xgb)

<Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1782770030542, experiment_id='2', last_update_time=1782770030542, lifecycle_stage='active', name='car_price_xgboost', tags={}, trace_location=None, workspace='default'>

In [54]:
# Optuna objective function for XGBoost
# Each trial proposes a set of hyperparameters, trains XGBoost on X_train, and returns the RMSE on X_val
# Optuna minimises this value across trials

def xgb_objective(trial: optuna.Trial) -> float:
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 600), # min and max
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.01, 5.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.01, 5.0, log=True),
        "random_state": 42,
        "verbosity": 0,
    }

    model = xgb.XGBRegressor(**params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    y_pred = model.predict(X_val)
    return root_mean_squared_error(y_val, y_pred)

In [55]:
# Run the Optuna study inside a parent MLflow run
# MLflowCallback logs each trial as a nested run automatically

mlflc = MLflowCallback(
    tracking_uri=mlflow.get_tracking_uri(),
    metric_name="rmse",
    mlflow_kwargs={"nested": True}
)

with mlflow.start_run(run_name="xgboost_optuna_tuning") as parent_run:

    print("🚀 Starting XGBoost Optuna study (20 trials)...")

    study_xgb = optuna.create_study(
        study_name="xgb_car_price_study",
        direction="minimize",
        storage="sqlite:///../data/optuna_study.db",
        load_if_exists=True
    )
    study_xgb.optimize(xgb_objective, n_trials=20, callbacks=[mlflc])

    print(f"✅ Best trial RMSE: £{study_xgb.best_value:,.0f}")
    print(f" Best params: {study_xgb.best_params}")

    # Log tags on the parent run
    mlflow.set_tags({
        "optimizer_engine": "optuna",
        "model_family": "xgboost",
        "project": "car_price_mlops",
        "environment": "dev"
    })

    # Log Optuna visualisation plots as HTML artifacts
    with tempfile.TemporaryDirectory() as tmpdir:
        vis.plot_optimization_history(study_xgb).write_html(
            os.path.join(tmpdir, "optimization_history.html"))
        vis.plot_param_importances(study_xgb).write_html(
            os.path.join(tmpdir, "param_importances.html"))
        vis.plot_parallel_coordinate(study_xgb).write_html(
            os.path.join(tmpdir, "parallel_coordinate.html"))
        mlflow.log_artifacts(tmpdir, artifact_path="optuna_visualizations")

    xgb_parent_run_id = parent_run.info.run_id

mlflow.end_run()

🚀 Starting XGBoost Optuna study (20 trials)...


[I 2026-06-29 23:18:59,866] Using an existing study with name 'xgb_car_price_study' instead of creating a new one.
[I 2026-06-29 23:19:08,361] Trial 20 finished with value: 2301.488037109375 and parameters: {'n_estimators': 199, 'max_depth': 8, 'learning_rate': 0.06311031534152577, 'subsample': 0.8582537188589185, 'colsample_bytree': 0.7198415470332258, 'reg_alpha': 0.09730984992538251, 'reg_lambda': 0.6059609491705564}. Best is trial 16 with value: 2264.53564453125.


🏃 View run 20 at: http://localhost:8080/#/experiments/3/runs/7e28be0c56e7461ca7c78233c635c2b9
🧪 View experiment at: http://localhost:8080/#/experiments/3


[I 2026-06-29 23:19:16,047] Trial 21 finished with value: 2324.62353515625 and parameters: {'n_estimators': 258, 'max_depth': 9, 'learning_rate': 0.04229701814243779, 'subsample': 0.7853405720797094, 'colsample_bytree': 0.8663186792619215, 'reg_alpha': 0.17068034537931048, 'reg_lambda': 1.9070690695172718}. Best is trial 16 with value: 2264.53564453125.


🏃 View run 21 at: http://localhost:8080/#/experiments/3/runs/7d74cd7f407b4acb880defd27433ee71
🧪 View experiment at: http://localhost:8080/#/experiments/3


...

🏃 View run 37 at: http://localhost:8080/#/experiments/3/runs/bb9eec1582464031b86854836f4c8ce4
🧪 View experiment at: http://localhost:8080/#/experiments/3


[I 2026-06-29 23:22:24,654] Trial 38 finished with value: 2410.15673828125 and parameters: {'n_estimators': 475, 'max_depth': 7, 'learning_rate': 0.019201963860411026, 'subsample': 0.8900656131086949, 'colsample_bytree': 0.6536054541129875, 'reg_alpha': 0.017104831597688868, 'reg_lambda': 0.025318070536005044}. Best is trial 28 with value: 2249.3623046875.


🏃 View run 38 at: http://localhost:8080/#/experiments/3/runs/309b1bad380d4bd7b8f8512810dc753b
🧪 View experiment at: http://localhost:8080/#/experiments/3


[I 2026-06-29 23:22:35,098] Trial 39 finished with value: 2260.334228515625 and parameters: {'n_estimators': 570, 'max_depth': 8, 'learning_rate': 0.07486011568441939, 'subsample': 0.949472526387756, 'colsample_bytree': 0.5631580185684067, 'reg_alpha': 0.010353652140334627, 'reg_lambda': 0.0314881961481228}. Best is trial 28 with value: 2249.3623046875.


🏃 View run 39 at: http://localhost:8080/#/experiments/3/runs/fb0d92b7534e46d484f186ed58708cef
🧪 View experiment at: http://localhost:8080/#/experiments/3
✅ Best trial RMSE: £2,249
 Best params: {'n_estimators': 400, 'max_depth': 9, 'learning_rate': 0.04402808064279601, 'subsample': 0.8974537652767574, 'colsample_bytree': 0.6487002595566211, 'reg_alpha': 0.02178907084709904, 'reg_lambda': 0.0933295479478435}
🏃 View run xgboost_optuna_tuning at: http://localhost:8080/#/experiments/2/runs/6f9385466ad441e8b4c9e2d67ab64605
🧪 View experiment at: http://localhost:8080/#/experiments/2


In [56]:
# Retrain with the best hyperparameters and log the final model

with mlflow.start_run(
    experiment_id=experiment_id_xgb,
    run_name="xgboost_best_model"
) as best_run:

    best_xgb = xgb.XGBRegressor(**study_xgb.best_params, random_state=42, verbosity=0)
    best_xgb.fit(X_train, y_train)
    y_pred_val = best_xgb.predict(X_val)

    # Evaluation Metrics
    xgb_rmse = root_mean_squared_error(y_val, y_pred_val)
    xgb_mae = mean_absolute_error(y_val, y_pred_val)
    xgb_r2 = r2_score(y_val, y_pred_val)

    mlflow.log_params(study_xgb.best_params)
    mlflow.log_metric("rmse", xgb_rmse)
    mlflow.log_metric("mae", xgb_mae)
    mlflow.log_metric("r2", xgb_r2)

    mlflow.set_tags({
        "model_family": "xgboost",
        "project": "car_price_mlops",
        "environment": "dev"
    })

    signature = infer_signature(X_train, y_pred_val)
    xgb_model_info = mlflow.xgboost.log_model(
        xgb_model=best_xgb,
        name="xgboost",
        signature=signature,
        input_example=X_train.iloc[[0]]
    )

    xgb_run_id = best_run.info.run_id

mlflow.end_run()

print(f"\n✅ XGBoost best model logged to '{EXPERIMENT_NAME_XGB}'.")
print(f"RMSE : £{xgb_rmse:,.0f}  (baseline was £4,404)")
print(f"MAE  : £{xgb_mae:,.0f}")
print(f"R²   : {xgb_r2:.4f}")
print(f"\nRun ID : {xgb_run_id}")
print("View   : http://localhost:8080")


2026/06/29 23:24:28 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run xgboost_best_model at: http://localhost:8080/#/experiments/2/runs/240dc8012306416ea67c622936363185
🧪 View experiment at: http://localhost:8080/#/experiments/2

✅ XGBoost best model logged to 'car_price_xgboost'.
RMSE : £2,249  (baseline was £4,404)
MAE  : £1,276
R²   : 0.9447

Run ID : 240dc8012306416ea67c622936363185
View   : http://localhost:8080


### Part 2B — LightGBM with Optuna Hyperparameter Tuning

We repeat the same Optuna tuning process for **LightGBM**, a gradient boosting framework developed by Microsoft that is generally faster to train than XGBoost and often achieves comparable or better results on tabular data.
 
The structure is identical to the XGBoost step:
 
1. **Tuning** — run 20 Optuna trials optimising RMSE on the validation set, each logged
   as a nested run under the parent run `lgbm_optuna_tuning`.
2. **Best model** — retrain LightGBM with the best hyperparameters and log it as a
   separate run named `lgbm_best_model` with metrics, tags, artifact, and Optuna plots.

At the end we will have three logged models: Linear Regression, XGBoost, and LightGBM, ready to be compared side by side in the next step.

In [57]:
EXPERIMENT_NAME_LGBM = "car_price_lgbm"
experiment_id_lgbm = get_or_create_experiment(EXPERIMENT_NAME_LGBM)
mlflow.set_experiment(experiment_id=experiment_id_lgbm)

<Experiment: artifact_location='mlflow-artifacts:/4', creation_time=1782770271606, experiment_id='4', last_update_time=1782770271606, lifecycle_stage='active', name='car_price_lgbm', tags={}, trace_location=None, workspace='default'>

In [58]:
# Optuna objective function for LightGBM

def lgbm_objective(trial: optuna.Trial) -> float:
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 600),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 20, 300),
        "lambda_l1": trial.suggest_float("lambda_l1", 0.01, 5.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 0.01, 5.0, log=True),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.5, 1.0),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "random_state": 42,
        "verbosity": -1,
    }

    model = LGBMRegressor(**params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)])
    y_pred = model.predict(X_val)
    return root_mean_squared_error(y_val, y_pred)

In [78]:
for col in X_train.columns:
    if X_train[col].dtype == object:
        print(col, type(X_train[col].iloc[0]))

In [59]:
#  Run the Optuna study inside a parent MLflow run
mlflc = MLflowCallback(
    tracking_uri=mlflow.get_tracking_uri(),
    metric_name="rmse",
    mlflow_kwargs={"nested": True}
)

with mlflow.start_run(run_name="lgbm_optuna_tuning") as parent_run:

    print("🚀 Starting LightGBM Optuna study (20 trials)...")

    study_lgbm = optuna.create_study(
        study_name="lgbm_car_price_study",
        direction="minimize",
        storage="sqlite:///../data/optuna_study.db",
        load_if_exists=True
    )
    study_lgbm.optimize(lgbm_objective, n_trials=20, callbacks=[mlflc])

    print(f"✅ Best trial RMSE: £{study_lgbm.best_value:,.0f}")
    print(f"   Best params: {study_lgbm.best_params}")

    mlflow.set_tags({
        "optimizer_engine": "optuna",
        "model_family": "lgbm",
        "project": "car_price_mlops",
        "environment": "dev"
    })

    # Log Optuna visualisation plots as HTML artifacts
    with tempfile.TemporaryDirectory() as tmpdir:
        vis.plot_optimization_history(study_lgbm).write_html(
            os.path.join(tmpdir, "optimization_history.html"))
        vis.plot_param_importances(study_lgbm).write_html(
            os.path.join(tmpdir, "param_importances.html"))
        vis.plot_parallel_coordinate(study_lgbm).write_html(
            os.path.join(tmpdir, "parallel_coordinate.html"))
        mlflow.log_artifacts(tmpdir, artifact_path="optuna_visualizations")

    lgbm_parent_run_id = parent_run.info.run_id

mlflow.end_run()

🚀 Starting LightGBM Optuna study (20 trials)...


[I 2026-06-29 23:24:39,470] Using an existing study with name 'lgbm_car_price_study' instead of creating a new one.
[I 2026-06-29 23:24:43,509] Trial 20 finished with value: 2387.325765202719 and parameters: {'n_estimators': 330, 'max_depth': 9, 'learning_rate': 0.05036083657340093, 'num_leaves': 75, 'lambda_l1': 0.5487342344481194, 'lambda_l2': 0.16268976311974337, 'bagging_fraction': 0.6320958537935908, 'min_child_samples': 16}. Best is trial 8 with value: 2293.2131297509004.


🏃 View run 20 at: http://localhost:8080/#/experiments/5/runs/13a1fe4062e34bb98d38cf73d9c52efe
🧪 View experiment at: http://localhost:8080/#/experiments/5


[I 2026-06-29 23:24:48,866] Trial 21 finished with value: 2421.368891778347 and parameters: {'n_estimators': 262, 'max_depth': 8, 'learning_rate': 0.06993583385701846, 'num_leaves': 143, 'lambda_l1': 1.77557267517048, 'lambda_l2': 4.162482346924363, 'bagging_fraction': 0.7596056739342638, 'min_child_samples': 16}. Best is trial 8 with value: 2293.2131297509004.


🏃 View run 21 at: http://localhost:8080/#/experiments/5/runs/81d0590cc4ed4c52865b60d93879b481
🧪 View experiment at: http://localhost:8080/#/experiments/5


...

🏃 View run 37 at: http://localhost:8080/#/experiments/5/runs/a743c8190a5048859391e5ad01b073c2
🧪 View experiment at: http://localhost:8080/#/experiments/5


[I 2026-06-29 23:26:42,016] Trial 38 finished with value: 2560.762584704243 and parameters: {'n_estimators': 523, 'max_depth': 6, 'learning_rate': 0.08293892961890441, 'num_leaves': 279, 'lambda_l1': 0.06493213949375441, 'lambda_l2': 3.403422009908292, 'bagging_fraction': 0.5316768087755592, 'min_child_samples': 66}. Best is trial 33 with value: 2282.7918096338663.


🏃 View run 38 at: http://localhost:8080/#/experiments/5/runs/370bc730e65d48c6ab19c1e16361b86c
🧪 View experiment at: http://localhost:8080/#/experiments/5


[I 2026-06-29 23:26:48,867] Trial 39 finished with value: 2342.8804280769855 and parameters: {'n_estimators': 487, 'max_depth': 7, 'learning_rate': 0.22843907364650354, 'num_leaves': 279, 'lambda_l1': 0.04445678842947397, 'lambda_l2': 2.130538109713698, 'bagging_fraction': 0.5031421000374604, 'min_child_samples': 18}. Best is trial 33 with value: 2282.7918096338663.


🏃 View run 39 at: http://localhost:8080/#/experiments/5/runs/d2a4b6a4295447b89c5c5a626ea85b76
🧪 View experiment at: http://localhost:8080/#/experiments/5
✅ Best trial RMSE: £2,283
   Best params: {'n_estimators': 564, 'max_depth': 7, 'learning_rate': 0.11561603861149876, 'num_leaves': 283, 'lambda_l1': 0.34622441298128565, 'lambda_l2': 3.355083366765643, 'bagging_fraction': 0.7079139400426321, 'min_child_samples': 11}
🏃 View run lgbm_optuna_tuning at: http://localhost:8080/#/experiments/4/runs/c22aea328bcf4c8b9e87fdca2c878bdd
🧪 View experiment at: http://localhost:8080/#/experiments/4


In [60]:
# Retrain with the best hyperparameters and log the final model

with mlflow.start_run(
    experiment_id=experiment_id_lgbm,
    run_name="lgbm_best_model"
) as best_run:

    best_lgbm = LGBMRegressor(**study_lgbm.best_params, random_state=42, verbosity=-1)
    best_lgbm.fit(X_train, y_train)
    y_pred_val = best_lgbm.predict(X_val)

    # Evaluation Metrics
    lgbm_rmse = root_mean_squared_error(y_val, y_pred_val)
    lgbm_mae  = mean_absolute_error(y_val, y_pred_val)
    lgbm_r2 = r2_score(y_val, y_pred_val)

    mlflow.log_params(study_lgbm.best_params)
    mlflow.log_metric("rmse", lgbm_rmse)
    mlflow.log_metric("mae",  lgbm_mae)
    mlflow.log_metric("r2",   lgbm_r2)

    mlflow.set_tags({
        "model_family": "lgbm",
        "project": "car_price_mlops",
        "environment": "dev"
    })

    signature = infer_signature(X_train, y_pred_val)
    lgbm_model_info = mlflow.lightgbm.log_model(
        lgb_model=best_lgbm,
        name="lgbm",
        signature=signature,
        input_example=X_train.iloc[[0]]
    )

    lgbm_run_id = best_run.info.run_id

mlflow.end_run()

print(f"\n✅ LightGBM best model logged to '{EXPERIMENT_NAME_LGBM}'.")
print(f"RMSE : £{lgbm_rmse:,.0f}  (baseline was £4,404)")
print(f"MAE  : £{lgbm_mae:,.0f}")
print(f"R²   : {lgbm_r2:.4f}")
print(f"\nRun ID : {lgbm_run_id}")
print("View   : http://localhost:8080")



2026/06/29 23:27:09 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/29 23:27:32 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run lgbm_best_model at: http://localhost:8080/#/experiments/4/runs/7d6295afa6fd4214a61e94a43d42b649
🧪 View experiment at: http://localhost:8080/#/experiments/4

✅ LightGBM best model logged to 'car_price_lgbm'.
RMSE : £2,283  (baseline was £4,404)
MAE  : £1,303
R²   : 0.9431

Run ID : 7d6295afa6fd4214a61e94a43d42b649
View   : http://localhost:8080


### Part 3A: Model Comparison & Champion Selection
 
With all three models trained and logged, we now compare them side by side and register the best one in the **MLflow Model Registry**.
 
We use `mlflow.search_runs()` to pull the final model runs from each experiment into a single DataFrame and rank them by RMSE. The model with the lowest RMSE becomes the **Champion**: the one that will move forward to SHAP explainability and serving.
 
The two runners-up are registered as **Challengers**, they remain versioned and auditable in the registry in case we need to roll back or compare again in the future.
 
We use `MlflowClient` for the registry operations, following the same Champion/Challenger pattern from the practical notebooks:
- `mlflow.register_model()` —> registers the model artifact from a run URI
- `client.set_registered_model_tag()` —> tags the model with task type and project
- `client.set_registered_model_alias()` —> assigns the `Champion` or `Challenger` alias

In [61]:
client = MlflowClient()

# Pull the best run from each experiment
# We search for the single run named "*_best_model" in each experiment, which is where we logged the final retrained model and its test metrics.

def get_best_run(experiment_name: str, run_name: str) -> pd.Series:
    """Returns the run row for a named run inside an experiment."""
    exp = mlflow.get_experiment_by_name(experiment_name)
    runs = mlflow.search_runs(
        experiment_ids=[exp.experiment_id],
        filter_string=f"tags.mlflow.runName = '{run_name}'"
    )
    return runs.iloc[0]

run_lr   = get_best_run("car_price_linear_regression", "linear_regression_baseline")
run_xgb  = get_best_run("car_price_xgboost", "xgboost_best_model")
run_lgbm = get_best_run("car_price_lgbm", "lgbm_best_model")

In [62]:
# Comparison table

comparison = pd.DataFrame({
    "Model": ["Linear Regression", "XGBoost", "LightGBM"],
    "RMSE (£)": [
        run_lr["metrics.rmse"],
        run_xgb["metrics.rmse"],
        run_lgbm["metrics.rmse"],
    ],
    "MAE (£)": [
        run_lr["metrics.mae"],
        run_xgb["metrics.mae"],
        run_lgbm["metrics.mae"],
    ],
    "R²": [
        run_lr["metrics.r2"],
        run_xgb["metrics.r2"],
        run_lgbm["metrics.r2"],
    ],
    "Run ID": [
        run_lr["run_id"],
        run_xgb["run_id"],
        run_lgbm["run_id"],
    ],
}).sort_values("RMSE (£)").reset_index(drop=True)

comparison["RMSE (£)"] = comparison["RMSE (£)"].map("£{:,.0f}".format)
comparison["MAE (£)"]  = comparison["MAE (£)"].map("£{:,.0f}".format)
comparison["R²"]       = comparison["R²"].map("{:.4f}".format)

print("📊 Model Comparison (sorted by RMSE):\n")
print(comparison[["Model", "RMSE (£)", "MAE (£)", "R²"]].to_string(index=False))

📊 Model Comparison (sorted by RMSE):

            Model RMSE (£) MAE (£)     R²
          XGBoost   £2,249  £1,276 0.9447
         LightGBM   £2,283  £1,303 0.9431
Linear Regression   £4,491  £2,681 0.7796


In [63]:
# Register all three models in the Model Registry
# rank 0 = Champion, rank 1 and 2 = Challengers

ranks    = ["Champion", "Challenger_1", "Challenger_2"]
run_rows = [
    get_best_run("car_price_linear_regression", "linear_regression_baseline"),
    get_best_run("car_price_xgboost", "xgboost_best_model"),
    get_best_run("car_price_lgbm", "lgbm_best_model"),
]

# Sort by RMSE ascending so index 0 is the Champion
run_rows.sort(key=lambda r: r["metrics.rmse"])

MODEL_REGISTRY_NAME = "car_price_model"

registered_versions = {}

for rank, run_row in zip(ranks, run_rows):
    model_uri = f"runs:/{run_row['run_id']}/{ run_row['tags.mlflow.runName'].replace('_baseline','').replace('_best_model','') }"

    result = mlflow.register_model(model_uri, MODEL_REGISTRY_NAME)
    version = result.version

    # Tag the version
    client.set_model_version_tag(MODEL_REGISTRY_NAME, version, "validation_status", "approved")
    client.set_model_version_tag(MODEL_REGISTRY_NAME, version, "rank", rank)

    # Set alias
    client.set_registered_model_alias(MODEL_REGISTRY_NAME, rank, version)

    registered_versions[rank] = version
    print(f"\n✅ Registered v{version} as '{rank}'")
    print(f"Model  : {run_row['tags.mlflow.runName']}")
    print(f"RMSE   : £{run_row['metrics.rmse']:,.0f}")

# Tag the registered model itself
client.set_registered_model_tag(MODEL_REGISTRY_NAME, "task", "regression")
client.set_registered_model_tag(MODEL_REGISTRY_NAME, "project", "car_price_mlops")

print(f"\n🏆 Champion is version {registered_versions['Champion']} — view the registry at http://localhost:8080")

Registered model 'car_price_model' already exists. Creating a new version of this model...
2026/06/29 23:28:24 WARNING mlflow.tracking._model_registry.fluent: Run with id 240dc8012306416ea67c622936363185 has no artifacts at artifact path 'xgboost', registering model based on models:/m-b4b2e1393595434aa14b59661251a400 instead
2026/06/29 23:28:41 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: car_price_model, version 4
Created version '4' of model 'car_price_model'.



✅ Registered v4 as 'Champion'
Model  : xgboost_best_model
RMSE   : £2,249


Registered model 'car_price_model' already exists. Creating a new version of this model...
2026/06/29 23:28:45 WARNING mlflow.tracking._model_registry.fluent: Run with id 7d6295afa6fd4214a61e94a43d42b649 has no artifacts at artifact path 'lgbm', registering model based on models:/m-6a5a69f40997483b878d240105ab60fe instead
2026/06/29 23:28:46 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: car_price_model, version 5
Created version '5' of model 'car_price_model'.



✅ Registered v5 as 'Challenger_1'
Model  : lgbm_best_model
RMSE   : £2,283


Registered model 'car_price_model' already exists. Creating a new version of this model...
2026/06/29 23:28:47 WARNING mlflow.tracking._model_registry.fluent: Run with id 53429cff478c478b84563b645d919e1a has no artifacts at artifact path 'linear_regression', registering model based on models:/m-dbb7fd0fa1784eb8b3d5e75f8e7df87f instead
2026/06/29 23:28:47 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: car_price_model, version 6
Created version '6' of model 'car_price_model'.



✅ Registered v6 as 'Challenger_2'
Model  : linear_regression_baseline
RMSE   : £4,491

🏆 Champion is version 4 — view the registry at http://localhost:8080


### Part 3B: SHAP Explainability
 
We now know *what* the Champion model predicts, but not *why*. **SHAP** (SHapley Additive exPlanations) answers that by computing the contribution of each feature to every individual prediction, grounded in game theory.
 
We compute SHAP values on the held-out test set using the Champion model (LightGBM) and log four plots as MLflow artifacts under `shap_plots/`:
 
- **Summary bar plot**: global feature importances ranked by mean |SHAP value| —> which features matter most across all predictions.
- **Beeswarm plot**: shows both the magnitude *and direction* of each feature's impact —> e.g. whether high mileage pushes price up or down.
- **Waterfall plot**: a single-prediction deep-dive on the highest-error test example —> shows exactly how the model arrived at that specific price.
- **Dependence plot**: how the top feature interacts with the second most important one.

We also log the mean absolute SHAP value per feature as individual MLflow metrics (e.g. `shap_year`, `shap_mileage`) so they can be tracked and compared across model versions over time.

In [68]:
import matplotlib
matplotlib.use("Agg")  # non-interactive backend so plots save correctly in notebooks
import matplotlib.pyplot as plt

# Load the Champion model from the registry
champion_model = mlflow.xgboost.load_model(
    "models:/car_price_model@Champion"
)
print("✅ Champion model loaded from registry.")

# Step 2: Compute SHAP values on the test set
explainer = shap.TreeExplainer(champion_model)
shap_values = explainer(X_test)

feature_names = X_test.columns.tolist()
print(f"✅ SHAP values computed on {X_test.shape[0]:,} test rows.")

✅ Champion model loaded from registry.
✅ SHAP values computed on 15,195 test rows.


In [69]:
# Log all plots and metrics to MLflow

with mlflow.start_run(
    experiment_id=experiment_id_lgbm,
    run_name="lgbm_shap_explainability"
) as shap_run:

    with tempfile.TemporaryDirectory() as tmpdir:

        # Plot 1: Summary bar (global feature importance)
        plt.figure(figsize=(10, 6))
        shap.summary_plot(shap_values, X_test, plot_type="bar",
                          feature_names=feature_names, show=False)
        plt.tight_layout()
        plt.savefig(os.path.join(tmpdir, "shap_summary_bar.png"), dpi=150)
        plt.close()

        # Plot 2: Beeswarm (direction + magnitude)
        plt.figure(figsize=(10, 6))
        shap.summary_plot(shap_values, X_test,
                          feature_names=feature_names, show=False)
        plt.tight_layout()
        plt.savefig(os.path.join(tmpdir, "shap_beeswarm.png"), dpi=150)
        plt.close()

        # Plot 3: Waterfall for the highest-error prediction
        y_pred_test   = champion_model.predict(X_test)
        errors = np.abs(y_pred_test - y_test.values)
        worst_idx = int(np.argmax(errors))

        plt.figure(figsize=(10, 6))
        shap.waterfall_plot(shap_values[worst_idx], show=False)
        plt.tight_layout()
        plt.savefig(os.path.join(tmpdir, "shap_waterfall_worst.png"), dpi=150)
        plt.close()

        # Plot 4: Dependence plot for the top feature
        mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
        top_feature = feature_names[int(np.argmax(mean_abs_shap))]

        plt.figure(figsize=(10, 6))
        shap.dependence_plot(top_feature, shap_values.values, X_test,
                             feature_names=feature_names, show=False)
        plt.tight_layout()
        plt.savefig(os.path.join(tmpdir, "shap_dependence_top.png"), dpi=150)
        plt.close()

        # Log all plots as artifacts
        mlflow.log_artifacts(tmpdir, artifact_path="shap_plots")

    # Log mean |SHAP| per feature as individual MLflow metrics
    for fname, shap_importance in zip(feature_names, mean_abs_shap):
        mlflow.log_metric(f"shap_{fname}", float(shap_importance))

    mlflow.set_tags({
        "model_family": "lgbm",
        "project": "car_price_mlops",
        "step": "explainability"
    })

    shap_run_id = shap_run.info.run_id

mlflow.end_run()


🏃 View run lgbm_shap_explainability at: http://localhost:8080/#/experiments/4/runs/a6b25cdeaff24b2c8f70a30836070b41
🧪 View experiment at: http://localhost:8080/#/experiments/4


MemoryError: Unable to allocate 1.72 GiB for an array with shape (15195, 15195) and data type float64

In [ ]:
# Top 5 most important features

importance_df = pd.DataFrame({
    "feature": feature_names,
    "mean_|SHAP|": mean_abs_shap
}).sort_values("mean_|SHAP|", ascending=False).reset_index(drop=True)

print("\n📊 Top 10 features by mean |SHAP value|:\n")
print(importance_df.head(10).to_string(index=False))
print(f"\nRun ID : {shap_run_id}")
print("View plots at http://localhost:8080 → car_price_lgbm → lgbm_shap_explainability → Artifacts → shap_plots/")


📊 Top 10 features by mean |SHAP value|:

         feature  mean_|SHAP|
     engine_size  3130.516811
            year  2098.717100
         mileage  1627.802355
           brand  1361.327543
             mpg   994.705699
    transmission   892.514346
      model_name   863.607580
       fuel_type   372.472281
         car_age   226.290832
mileage_per_year   196.866359

Run ID : f2631618d50047fba2911a50ccc2455d
View plots at http://localhost:8080 → car_price_lgbm → lgbm_shap_explainability → Artifacts → shap_plots/


The SHAP values reveal which features drive used car prices most strongly in our Champion
model (LightGBM). The key findings are:
 
- **`engine_size`** is the single most influential feature (mean |SHAP| £3,131). Larger engines consistently push predicted prices upward — a 3.0L car commands significantly more than a 1.0L equivalent, all else being equal.
- **`year`** is the second strongest driver (£2,099). Newer cars are worth more, and the model has learned a strong positive relationship between manufacture year and price.
- **`mileage`** has a large negative impact (£1,628). Higher odometer readings consistently depress predicted prices — each additional mile driven reduces the model's price estimate.
- **`brand`** contributes £1,361 on average, reflecting that premium brands (BMW, Audi, Mercedes) command a substantial price premium over mass-market alternatives.
- **`mpg`** and **`transmission`** are meaningful but secondary (£995 and £893) -> automatic and semi-automatic vehicles tend to be priced higher than manual equivalents.
- Engineered features **`car_age`** and **`mileage_per_year`** contribute less than raw `year` and `mileage` individually, suggesting the model extracts most of the signal directly from the original features.

These findings are intuitive and consistent with the used car market: engine size, age, and mileage are the three pillars of vehicle valuation.

### Part 3C: Final Evaluation with `mlflow.evaluate()`
 
As a final standardised evaluation step, we run `mlflow.evaluate()` on the held-out **test set** — data the Champion model has never seen during training or tuning.
 
`mlflow.evaluate()` is MLflow's built-in evaluation framework. For regressors it automatically computes and logs:
- **RMSE, MAE, R², Max Error** — the standard regression metrics
- **Residual plots** — predicted vs actual and residuals vs predicted, saved as artifacts

This gives us a single, reproducible evaluation record on the true held-out test set, separate from the validation metrics used during Optuna tuning. It is the number we report as the final model performance.

In [ ]:
# mlflow.evaluate() requires the target
# column to be included in the same DataFrame as the features
eval_data = X_test.copy()
eval_data["price"] = y_test.values
 
with mlflow.start_run(
    experiment_id=experiment_id_lgbm,
    run_name="lgbm_final_evaluation"
) as eval_run:

    # Log the champion model artifact so evaluate() can load it
    signature = infer_signature(X_test, champion_model.predict(X_test))
    model_info = mlflow.lightgbm.log_model(
        lgb_model=champion_model,
        name="champion_model",
        signature=signature,
        input_example=X_test.iloc[[0]]
    )

    # Run automated evaluation — generates metrics + residual plots automatically
    eval_result = mlflow.evaluate(
        model=model_info.model_uri,
        data=eval_data,
        targets="price",
        model_type="regressor",
        evaluators=["default"]
    )

    mlflow.set_tags({
        "model_family": "lgbm",
        "project": "car_price_mlops",
        "step": "final_evaluation",
        "dataset": "test_set"
    })

    eval_run_id = eval_run.info.run_id

mlflow.end_run()

2026/06/26 17:58:12 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-bc9ecb819323452484fd4ad3935b55ea
2026/06/26 17:58:12 INFO mlflow.tracking.fluent: Use `mlflow.set_active_model` to set the active model to a different one if needed.
2026/06/26 17:58:12 INFO mlflow.models.evaluation.default_evaluator: Testing metrics on first row...
Background dataset has 2000 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=2000 when initializing the masker.
2026/06/26 17:58:12 INFO mlflow.models.evaluation.evaluators.shap: Shap explainer PermutationExplainer is used.
2026/06/26 17:58:14 WARNING mlflow.models.evaluation.evaluators.shap: Shap evaluation failed. Reason: MlflowException("Failed to enforce schema of data '      brand  model_name    year  transmission  mileage  fuel_type    tax  \\\n0       8.0        75.0  2019.0           2.0   2865.0        5.0  145.0   \n1       6.0       206.0  2014.

🏃 View run lgbm_final_evaluation at: http://localhost:8080/#/experiments/4/runs/2d3788ab4b4546bc99356fcb19d026d9
🧪 View experiment at: http://localhost:8080/#/experiments/4


In [ ]:
# print final metrics

print("✅ Final evaluation on held-out test set complete.\n")
print(f"{'Metric':<30} {'Value':>12}")
print(f"{'-'*44}")
for metric, value in eval_result.metrics.items():
    if isinstance(value, float):
        print(f"   {metric:<30} {value:>12.4f}")

print(f"\n   Run ID : {eval_run_id}")
print("View residual plots at http://localhost:8080 → car_price_lgbm → lgbm_final_evaluation → Artifacts")

✅ Final evaluation on held-out test set complete.

Metric                                Value
--------------------------------------------
   mean_absolute_error               1363.1493
   mean_squared_error             4444839.6370
   root_mean_squared_error           2108.2788
   mean_on_target                   16859.9422
   r2_score                             0.9499
   max_error                        20679.3004
   mean_absolute_percentage_error       0.0867

   Run ID : 2d3788ab4b4546bc99356fcb19d026d9
View residual plots at http://localhost:8080 → car_price_lgbm → lgbm_final_evaluation → Artifacts
